# K-Fold training — 4 model (dataset dedupe_gui, cosine) — tanpa MobileNetV3-Large TV

Notebook ini adalah **notebook baru** (tidak menimpa `Cosine_DedupeGUI_5Models_Comparison.ipynb`). Tujuannya menjawab kekurangan run sebelumnya:

- Run sebelumnya memakai **satu split 8:1:1** → hanya satu angka point-estimate per model. Skor papan atas **rapat** (RepViT-M0.9 85,64% vs StarNet-S2 85,64% — seri di 2 desimal) sehingga pemenang tidak bisa diputuskan secara meyakinkan, dan tidak ada ukuran variansi.
- Solusinya: **Stratified K-Fold cross-validation** (default 3-fold, seed 42) di atas **dataset kurasi dedupe_gui yang sama** (11.236 gambar). Setiap model dilaporkan sebagai **mean ± std** akurasi / macro-F1 / 3-class accuracy lintas fold, plus tabel per-fold dan plot error-bar.
- **MobileNetV3-Large TV dibuang** dari perbandingan: pada run single-split ia peringkat 4/5 (83,33%, macro-F1 0,8246) dengan 4,24M parameter — bukan kandidat teratas maupun paling hemat. Fokus 4 model: **RepViT-M0.9, StarNet-S2, MobileNetV2, MobileNetV3-Small** (2 kandidat teratas + baseline acuan + kandidat ultra-ringan).

**Resep training per model (sama seperti varian B·cosine notebook asli):** SGD momentum 0.9, **CosineAnnealingLR(T_max=NUM_EPOCHS)**, label smoothing 0.1, weight decay 1e-4, augmentasi `RandomResizedCrop`+`RandomHorizontalFlip`, input 224×224 + ImageNet norm, batch 42, 50 epoch, early stopping (patience 10, min epoch 10). LR per model: RepViT/StarNet-S2 **2e-3**, MobileNetV2/MobileNetV3-Small **1e-3**.

**Desain fold:** StratifiedKFold(n_splits=5, shuffle, seed 42) berdasar label 30 kelas pada seluruh frame kurasi. Tiap fold: 1/K sebagai **test**, sisa (K-1)/K dipecah lagi stratified 90/10 menjadi **train/val** (untuk K=3: train ±60% / val ±6,7%) untuk early stopping & pemilihan checkpoint. Checkpoint dipilih dari **val fold**, dievaluasi pada **test fold yang tidak pernah dilihat saat training**.

> Artefak run ini memakai prefix `*_kfold_dedupe_gui` dan file `kfold_dedupe_gui_*` agar **tidak menimpa** hasil run sebelumnya (`*_cosine_dedupe_gui`).


## 1. Setup dan dependensi

In [ ]:
# ---------------------------------------------------------------------------
# Auto-install dependencies. Packages that are already installed are untouched.
# ---------------------------------------------------------------------------
from __future__ import annotations

import importlib.util
import subprocess
import sys


def module_available(name: str) -> bool:
    return importlib.util.find_spec(name) is not None


def pip_install(*packages: str, index_url: str | None = None) -> bool:
    command = [sys.executable, "-m", "pip", "install", "-q"]
    if index_url:
        command += ["--index-url", index_url]
    command += list(packages)
    print("    $", " ".join(command))
    try:
        return subprocess.run(command, check=False).returncode == 0
    except Exception as exc:
        print(f"[warn] pip install failed for {' '.join(packages)}: {exc}")
        return False


def ensure(packages: dict[str, str]) -> list[str]:
    missing = [pkg for pkg, mod in packages.items() if not module_available(mod)]
    if not missing:
        return []
    print(f"[setup] installing: {', '.join(missing)}")
    for package in missing:
        pip_install(package)
    still_missing = [pkg for pkg in missing if not module_available(packages[pkg])]
    if still_missing:
        print("[warn] could not install: " + ", ".join(still_missing))
    return still_missing


ensure({
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "scikit-learn": "sklearn",
    "Pillow": "PIL",
    "tqdm": "tqdm",
    "ipywidgets": "ipywidgets",
    "timm": "timm",
})

if not module_available("torch") or not module_available("torchvision"):
    print("[setup] installing torch and torchvision")
    installed = pip_install(
        "torch", "torchvision", index_url="https://download.pytorch.org/whl/cpu"
    )
    if not installed and (not module_available("torch") or not module_available("torchvision")):
        print("[setup] retrying with the default PyPI index")
        pip_install("torch", "torchvision")

print("[setup] dependency check done")


In [ ]:
from __future__ import annotations

import copy
import csv
import hashlib
import json
import math
import os
import random
import sys
import time
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile
try:
    from ipywidgets import IProgress  # noqa: F401
    from tqdm.auto import tqdm
except Exception:
    from tqdm import tqdm
from IPython.display import display

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    top_k_accuracy_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 200,
    "axes.unicode_minus": False,
})

SEED = 42


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything()


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "outputs").is_dir() and (candidate / "Datasets").is_dir():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd().resolve())
OUTPUT_DIR = REPO_ROOT / "outputs"
SPLIT_DIR = OUTPUT_DIR / "splits"
MODEL_DIR = OUTPUT_DIR / "models"
RESULT_DIR = OUTPUT_DIR / "results"
FIGURE_DIR = OUTPUT_DIR / "figures"
for _d in (OUTPUT_DIR, SPLIT_DIR, MODEL_DIR, RESULT_DIR, FIGURE_DIR):
    _d.mkdir(parents=True, exist_ok=True)

DATA_ROOT = Path(
    os.environ.get("TRASH_DATA_DIR", REPO_ROOT / "Datasets" / "images" / "images")
).resolve()

DECISIONS_CSV = RESULT_DIR / "duplicate_groups_with_decision.csv"
GROUPS_MD5_CSV = RESULT_DIR / "duplicate_groups_md5.csv"

# Protocol defaults (mengikuti notebook asli; LR per-model di-override di MODEL_SPECS)
IMG_SIZE = 224
BATCH_SIZE = 42
NUM_EPOCHS = 50
LEARNING_RATE = 1e-3
MOMENTUM = 0.9
STEP_SIZE = 7
GAMMA = 0.1
NUM_WORKERS = 0
PRETRAINED = os.environ.get("TRASH_REPRO_PRETRAINED", "1") == "1"
TEST_BATCH_SIZE = 128
EARLY_STOP_PATIENCE = 10
EARLY_STOP_MIN_EPOCHS = 10

QUICK_TEST = os.environ.get("TRASH_REPRO_QUICK", "0") == "1"
if QUICK_TEST:
    NUM_EPOCHS = 2
    BATCH_SIZE = 16
    PRETRAINED = False
    EARLY_STOP_PATIENCE = 2
    EARLY_STOP_MIN_EPOCHS = 1

# Jumlah fold: default 3 (12 training), bisa dioverride via env KFOLD_SPLITS (mis. 5 untuk run lebih teliti).
N_SPLITS = int(os.environ.get("KFOLD_SPLITS", "3"))
if QUICK_TEST:
    N_SPLITS = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Python           : {sys.version.split()[0]}")
print(f"PyTorch          : {torch.__version__}")
print(f"Device           : {DEVICE}")
print(f"Repo root        : {REPO_ROOT}")
print(f"Data root        : {DATA_ROOT} (exists={DATA_ROOT.exists()})")
print(f"Quick test       : {QUICK_TEST}")
print(f"Pretrained       : {PRETRAINED}")
print(f"K folds          : {N_SPLITS}")
print(f"Epochs           : {NUM_EPOCHS}")


## 2. Dataset kurasi dedupe_gui (sama seperti run 5-model sebelumnya)

Membangun set gambar final dari keputusan `dedupe_gui.py` (sama persis seperti notebook 5-model): semua file di disk dikurangi anggota grup duplikat, ditambah gambar berkeputusan **keep**. Bedanya dengan notebook sebelumnya: di sini **tidak** dipakai split 8:1:1 tunggal — seluruh frame kurasi dipakai untuk K-Fold. Manifest `dedupe_gui_*.csv` tetap ditulis ulang agar konsisten bila belum ada.

In [ ]:
CLASS_NAMES = [
    "aerosol_cans", "aluminum_food_cans", "aluminum_soda_cans",
    "cardboard_boxes", "cardboard_packaging", "clothing",
    "coffee_grounds", "disposable_plastic_cutlery", "eggshells",
    "food_waste", "glass_beverage_bottles", "glass_cosmetic_containers",
    "glass_food_jars", "magazines", "newspaper", "office_paper",
    "paper_cups", "plastic_cup_lids", "plastic_detergent_bottles",
    "plastic_food_containers", "plastic_shopping_bags", "plastic_soda_bottles",
    "plastic_straws", "plastic_trash_bags", "plastic_water_bottles",
    "shoes", "steel_food_cans", "styrofoam_cups",
    "styrofoam_food_containers", "tea_bags",
]
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)

MAJOR_NAMES = ["Kitchen waste", "Recyclable waste", "Other waste"]
MAJOR_TO_IDX = {name: i for i, name in enumerate(MAJOR_NAMES)}

CLASS_TO_MAJOR = {
    "coffee_grounds": "Kitchen waste", "eggshells": "Kitchen waste",
    "food_waste": "Kitchen waste", "tea_bags": "Kitchen waste",
    "aerosol_cans": "Recyclable waste", "aluminum_food_cans": "Recyclable waste",
    "aluminum_soda_cans": "Recyclable waste", "cardboard_boxes": "Recyclable waste",
    "cardboard_packaging": "Recyclable waste", "clothing": "Recyclable waste",
    "glass_beverage_bottles": "Recyclable waste", "glass_cosmetic_containers": "Recyclable waste",
    "glass_food_jars": "Recyclable waste", "magazines": "Recyclable waste",
    "newspaper": "Recyclable waste", "office_paper": "Recyclable waste",
    "paper_cups": "Recyclable waste", "plastic_detergent_bottles": "Recyclable waste",
    "plastic_food_containers": "Recyclable waste", "plastic_shopping_bags": "Recyclable waste",
    "plastic_soda_bottles": "Recyclable waste", "plastic_water_bottles": "Recyclable waste",
    "shoes": "Recyclable waste", "steel_food_cans": "Recyclable waste",
    "disposable_plastic_cutlery": "Other waste", "plastic_cup_lids": "Other waste",
    "plastic_straws": "Other waste", "plastic_trash_bags": "Other waste",
    "styrofoam_cups": "Other waste", "styrofoam_food_containers": "Other waste",
}
assert not (set(CLASS_NAMES) - set(CLASS_TO_MAJOR)), "Missing major-class mapping"
print(f"Classes: {NUM_CLASSES}")


In [ ]:
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}


def scan_all_images(root: Path) -> set[str]:
    paths = set()
    for dirpath, _, files in os.walk(root):
        for f in files:
            if os.path.splitext(f)[1].lower() in IMAGE_EXTENSIONS:
                p = Path(dirpath) / f
                try:
                    paths.add(p.relative_to(root).as_posix())
                except ValueError:
                    pass
    return paths


def infer_source_group(rel_path: str) -> str:
    parts = Path(rel_path).parts
    for p in parts:
        if p in {"default", "real_world"}:
            return p
    return "unknown"


assert DATA_ROOT.exists(), f"Dataset folder not found: {DATA_ROOT}. Set TRASH_DATA_DIR."
missing_dirs = [name for name in CLASS_NAMES if not (DATA_ROOT / name).is_dir()]
assert not missing_dirs, f"Missing class folders under {DATA_ROOT}: {missing_dirs}"

all_disk = scan_all_images(DATA_ROOT)
print(f"Images on disk: {len(all_disk):,}")

group_members = set()
if GROUPS_MD5_CSV.exists():
    with GROUPS_MD5_CSV.open(newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            group_members.add(row["rel_path"])
elif DECISIONS_CSV.exists():
    with DECISIONS_CSV.open(newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            group_members.add(row["rel_path"])
print(f"In duplicate groups: {len(group_members):,}")

kept, removed = set(), set()
if DECISIONS_CSV.exists():
    with DECISIONS_CSV.open(newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            (kept if row.get("decision") == "keep" else removed).add(row["rel_path"])
    print(f"dedupe_gui decisions - keep: {len(kept):,}, remove: {len(removed):,}")
else:
    print(f"[warn] {DECISIONS_CSV} not found; using all disk images.")

if group_members:
    curated_rel = (all_disk - group_members) | kept
else:
    curated_rel = set(all_disk)
curated_rel = {p for p in curated_rel if (DATA_ROOT / p).exists() and Path(p).parts[0] in CLASS_NAMES}
print(f"Curated images (dedupe_gui): {len(curated_rel):,}")

records = []
for rel in sorted(curated_rel):
    class_name = Path(rel).parts[0]
    source_group = infer_source_group(rel)
    records.append({
        "abs_path": str((DATA_ROOT / rel).resolve()),
        "rel_path": rel,
        "class_name": class_name,
        "source_group": source_group,
    })

df = pd.DataFrame(records)
assert not df.empty, "No curated images found."
df["label"] = df["class_name"].map(CLASS_TO_IDX).astype(int)
df = df.sort_values(["class_name", "rel_path"]).reset_index(drop=True)

# Tulis manifest referensi (tidak dipakai untuk training K-Fold, hanya arsip konsistensi).
SPLIT_ALL = SPLIT_DIR / "dedupe_gui_all.csv"
df.to_csv(SPLIT_ALL, index=False)
print(f"Curated frame: {len(df):,} images | classes: {df['class_name'].nunique()}")
print(f"Source groups: {df['source_group'].value_counts().to_dict()}")
print(f"Manifest ditulis: {SPLIT_ALL}")
assert not df["rel_path"].duplicated().any(), "Duplicate rel_path in curated manifest."


## 3. Desain K-Fold (pengganti split tunggal 8:1:1)

StratifiedKFold berdasar label 30 kelas pada seluruh frame kurasi. Tiap fold luar: 1 bagian = test, 4 bagian = train+val yang dipecah lagi stratified 90/10 menjadi train/val untuk early stopping. Total run = 4 model × N_SPLITS fold (default 3 → 12 training). Untuk smoke test (`TRASH_REPRO_QUICK=1`) otomatis N_SPLITS=2 dengan subsample kecil.

In [ ]:
if QUICK_TEST:
    df = (df.groupby("class_name", group_keys=False)
            .apply(lambda g: g.sample(n=min(30, len(g)), random_state=SEED))
            .reset_index(drop=True))
    print(f"[quick] subsampled frame: {len(df):,} images")

y_all = df["label"].to_numpy()
min_per_class = pd.Series(y_all).value_counts().min()
assert min_per_class >= N_SPLITS, (
    f"Kelas terkecil hanya {min_per_class} gambar < N_SPLITS={N_SPLITS}. "
    "Turunkan KFOLD_SPLITS (mis. 2)."
)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = list(skf.split(np.zeros(len(df)), y_all))
print(f"Folds: {N_SPLITS} | images: {len(df):,}")
for i, (trv_idx, te_idx) in enumerate(folds):
    te_n = len(te_idx)
    trv = df.iloc[trv_idx]
    # cek liputan kelas pada tiap sisi fold
    assert trv["class_name"].nunique() == NUM_CLASSES
    assert df.iloc[te_idx]["class_name"].nunique() == NUM_CLASSES
    print(f"  fold {i}: trainval {len(trv_idx):,} | test {te_n:,} "
          f"| test share {te_n/len(df):.1%}")

# Simpan assignment fold agar run bisa direplikasi / diaudit.
assign = df[["rel_path", "class_name", "source_group", "label"]].copy()
assign["fold_test"] = -1
for i, (_, te_idx) in enumerate(folds):
    assign.loc[te_idx, "fold_test"] = i
assign_path = RESULT_DIR / "kfold_dedupe_gui_fold_assignment.csv"
assign.to_csv(assign_path, index=False)
print(f"Saved fold assignment: {assign_path}")


## 4. Transform, dataset, dan loader

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class TrashDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, root: Path, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.root = Path(root)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int):
        row = self.frame.iloc[index]
        with Image.open(self.root / row["rel_path"]) as img:
            image = img.convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, int(row["label"])


def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_loader(frame: pd.DataFrame, training: bool, batch_size: int, transform=None) -> DataLoader:
    dataset = TrashDataset(
        frame, DATA_ROOT,
        transform=transform if transform is not None else (train_transform if training else eval_transform),
    )
    drop_last = training and len(dataset) >= batch_size and len(dataset) % batch_size == 1
    generator = torch.Generator()
    generator.manual_seed(SEED)
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=training,
        num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
        drop_last=drop_last, persistent_workers=NUM_WORKERS > 0,
        worker_init_fn=seed_worker, generator=generator,
    )

print("Transform & loader utils siap.")


## 5. Definisi 4 model (tanpa MobileNetV3-Large TV)

Alasan TV dibuang: pada run single-split dedupe_gui ia peringkat 4/5 (83,33%, macro-F1 0,8246) dengan 4,24M parameter — bukan kandidat teratas maupun paling hemat. 4 model yang dipertahankan: 2 teratas (RepViT-M0.9, StarNet-S2), baseline acuan (MobileNetV2), dan kandidat ultra-ringan (MobileNetV3-Small). Config = varian B·cosine notebook asli masing-masing.

In [ ]:
import timm


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def build_mobilenet_v2(num_classes: int, pretrained: bool = True) -> nn.Module:
    weights = "IMAGENET1K_V1" if pretrained else None
    model = models.mobilenet_v2(weights=weights)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model


def build_mobilenet_v3_small(num_classes: int, pretrained: bool = True) -> nn.Module:
    weights = "IMAGENET1K_V1" if pretrained else None
    model = models.mobilenet_v3_small(weights=weights)
    model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)
    return model


def build_repvit_m0_9(num_classes: int, pretrained: bool = True) -> nn.Module:
    return timm.create_model(
        "repvit_m0_9.dist_450e_in1k", pretrained=pretrained, num_classes=num_classes
    )


def build_starnet_s2(num_classes: int, pretrained: bool = True) -> nn.Module:
    return timm.create_model(
        "starnet_s2.in1k", pretrained=pretrained, num_classes=num_classes
    )


# 4 model SAJA (tanpa mobilenetv3large_tv). Prefix *_kfold_dedupe_gui agar tidak menimpa run lama.
MODEL_SPECS = {
    "repvit_m0_9_in1k_kfold_dedupe_gui": {
        "label": "RepViT-M0.9 (cosine, kfold dedupe_gui)",
        "arch": "repvit_m0_9",
        "builder": build_repvit_m0_9,
        "params_m": 4.742428, "gmacs": 0.816535872,
        "learning_rate": 2e-3, "momentum": 0.9,
        "label_smoothing": 0.1, "weight_decay": 1e-4,
        "scheduler": "cosine", "class_weight": False,
        "augmentation": "randomresizedcrop+flip",
    },
    "starnet_s2_in1k_kfold_dedupe_gui": {
        "label": "StarNet-S2 (cosine, kfold dedupe_gui)",
        "arch": "starnet_s2",
        "builder": build_starnet_s2,
        "params_m": 3.432734, "gmacs": 0.543865344,
        "learning_rate": 2e-3, "momentum": 0.9,
        "label_smoothing": 0.1, "weight_decay": 1e-4,
        "scheduler": "cosine", "class_weight": False,
        "augmentation": "randomresizedcrop+flip",
    },
    "mobilenetv2_kfold_dedupe_gui": {
        "label": "MobileNetV2 (cosine, kfold dedupe_gui)",
        "arch": "mobilenet_v2",
        "builder": build_mobilenet_v2,
        "params_m": 2.262302, "gmacs": 0.299532672,
        "learning_rate": 1e-3, "momentum": 0.9,
        "label_smoothing": 0.1, "weight_decay": 1e-4,
        "scheduler": "cosine", "class_weight": False,
        "augmentation": "randomresizedcrop+flip",
    },
    "mobilenetv3small_kfold_dedupe_gui": {
        "label": "MobileNetV3-Small (cosine, kfold dedupe_gui)",
        "arch": "mobilenet_v3_small",
        "builder": build_mobilenet_v3_small,
        "params_m": 1.548606, "gmacs": 0.05551712,
        "learning_rate": 1e-3, "momentum": 0.9,
        "label_smoothing": 0.1, "weight_decay": 1e-4,
        "scheduler": "cosine", "class_weight": False,
        "augmentation": "randomresizedcrop+flip",
    },
}

for key, spec in MODEL_SPECS.items():
    print(
        f"{key:45s} | lr={spec['learning_rate']:<7g} | sched={spec['scheduler']:<6} | "
        f"ls={spec['label_smoothing']} | wd={spec['weight_decay']} | aug={spec['augmentation']}"
    )
print(f"\nTotal: {len(MODEL_SPECS)} model x {N_SPLITS} fold = {len(MODEL_SPECS)*N_SPLITS} training")


## 6. Utilitas training dan evaluasi (sama seperti notebook 5-model)

In [ ]:
def jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, Path):
        return str(obj)
    return obj


def load_checkpoint(path, device):
    try:
        return torch.load(path, map_location=device, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=device)
    except Exception:
        return torch.load(path, map_location=device, weights_only=False)


def train_model(
    model, train_loader, val_loader, criterion, optimizer, scheduler,
    epochs, device, checkpoint_path, history_path, model_name, arch,
    extra_config=None, full_model_path=None,
    early_stopping_patience=None, early_stop_min_epochs=0,
):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_acc = -1.0
    best_epoch = 0
    epochs_without_improvement = 0
    history = []
    dataloaders = {"train": train_loader, "val": val_loader}
    dataset_sizes = {phase: len(loader.dataset) for phase, loader in dataloaders.items()}

    for epoch in range(1, epochs + 1):
        print(f"Epoch {epoch}/{epochs}")
        epoch_start = time.time()
        epoch_losses, epoch_accs = {}, {}
        improved_this_epoch = False

        for phase in ["train", "val"]:
            model.train() if phase == "train" else model.eval()
            running_loss = 0.0
            running_corrects = 0
            loader = dataloaders[phase]
            progress = tqdm(total=dataset_sizes[phase], desc=f"{model_name} {phase} {epoch}/{epochs}", leave=False)
            for inputs, labels in loader:
                inputs = inputs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                optimizer.zero_grad()
                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    if phase == "train":
                        loss.backward()
                        optimizer.step()
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                progress.update(inputs.size(0))
            progress.close()

            if phase == "train" and scheduler is not None:
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = (running_corrects.double() / dataset_sizes[phase]).item()
            epoch_losses[phase] = float(epoch_loss)
            epoch_accs[phase] = float(epoch_acc)
            print(f"  {phase} loss {epoch_loss:.4f} acc {epoch_acc:.4f}")

            if phase == "val" and epoch_acc > best_val_acc:
                best_val_acc = float(epoch_acc)
                best_epoch = epoch
                improved_this_epoch = True
                best_model_wts = copy.deepcopy(model.state_dict())
                torch.save({
                    "arch": arch, "model_name": model_name, "state_dict": model.state_dict(),
                    "class_names": CLASS_NAMES, "major_names": MAJOR_NAMES,
                    "class_to_major": CLASS_TO_MAJOR, "img_size": IMG_SIZE,
                    "mean": IMAGENET_MEAN, "std": IMAGENET_STD,
                    "epoch": epoch, "val_acc": float(epoch_acc), "val_loss": float(epoch_loss),
                    "config": {"seed": SEED, "batch_size": BATCH_SIZE, "epochs": epochs,
                               "pretrained": PRETRAINED, **(extra_config or {})},
                }, checkpoint_path)

        epochs_without_improvement = 0 if improved_this_epoch else epochs_without_improvement + 1
        current_lr = optimizer.param_groups[0]["lr"]
        record = {
            "epoch": epoch, "train_loss": epoch_losses["train"], "train_acc": epoch_accs["train"],
            "val_loss": epoch_losses["val"], "val_acc": epoch_accs["val"],
            "lr": float(current_lr), "time_sec": float(time.time() - epoch_start), "best_val_acc": best_val_acc,
        }
        history.append(record)
        pd.DataFrame(history).to_csv(history_path, index=False)

        if (early_stopping_patience is not None and epoch >= early_stop_min_epochs
                and epochs_without_improvement >= early_stopping_patience):
            print(f"Early stopping at epoch {epoch} (no improvement for {early_stopping_patience} epochs).")
            break

    time_elapsed = time.time() - since
    print(f"Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s | best val {best_val_acc:.4f} @ {best_epoch}")
    model.load_state_dict(best_model_wts)
    if full_model_path is not None:
        torch.save(model, full_model_path)
    return history, best_val_acc, best_epoch


def aggregate_major_probs(probs: np.ndarray, class_names):
    major_probs = np.zeros((probs.shape[0], len(MAJOR_NAMES)), dtype=np.float64)
    for class_idx, class_name in enumerate(class_names):
        major_probs[:, MAJOR_TO_IDX[CLASS_TO_MAJOR[class_name]]] += probs[:, class_idx]
    return major_probs


@torch.no_grad()
def predict_loader(model, loader, criterion, device):
    model.eval()
    all_probs, all_labels = [], []
    total_loss = 0.0
    total_samples = 0
    for images, labels in tqdm(loader, desc="Evaluating", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        outputs = model(images)
        loss = criterion(outputs, labels)
        probs = torch.softmax(outputs, dim=1)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())
        total_loss += loss.item() * labels.size(0)
        total_samples += labels.size(0)
    probs = np.concatenate(all_probs, axis=0)
    labels = np.concatenate(all_labels, axis=0)
    return probs, labels, total_loss / max(total_samples, 1)


def evaluate_and_save(model, loader, criterion, device, frame, model_key):
    probs, labels, loss = predict_loader(model, loader, criterion, device)
    preds = probs.argmax(axis=1)
    report_dict = classification_report(
        labels, preds, labels=np.arange(NUM_CLASSES), target_names=CLASS_NAMES,
        output_dict=True, zero_division=0,
    )
    cm = confusion_matrix(labels, preds, labels=np.arange(NUM_CLASSES))
    per_class_acc = np.divide(cm.diagonal(), cm.sum(axis=1), out=np.zeros(NUM_CLASSES, dtype=np.float64),
                              where=cm.sum(axis=1) != 0)
    major_probs = aggregate_major_probs(probs, CLASS_NAMES)
    major_true = frame["class_name"].map(lambda c: MAJOR_TO_IDX[CLASS_TO_MAJOR[c]]).to_numpy()
    major_pred = np.array([MAJOR_TO_IDX[CLASS_TO_MAJOR[CLASS_NAMES[int(i)]]] for i in preds])
    major_prob_pred = major_probs.argmax(axis=1)
    major_report = classification_report(
        major_true, major_pred, labels=np.arange(len(MAJOR_NAMES)),
        target_names=MAJOR_NAMES, output_dict=True, zero_division=0,
    )
    major_cm = confusion_matrix(major_true, major_pred, labels=np.arange(len(MAJOR_NAMES)))
    pred_frame = frame[["rel_path", "class_name", "source_group"]].copy()
    pred_frame["true_label"] = labels
    pred_frame["pred_label"] = preds
    pred_frame["pred_class"] = [CLASS_NAMES[i] for i in preds]
    pred_frame["confidence"] = probs.max(axis=1)
    pred_frame["correct"] = (labels == preds).astype(int)
    pred_frame.to_csv(RESULT_DIR / f"{model_key}_predictions.csv", index=False)
    metrics = {
        "model": model_key, "split": "test", "num_test": int(len(labels)),
        "accuracy": float(accuracy_score(labels, preds)), "loss": float(loss),
        "top3_accuracy": float(top_k_accuracy_score(labels, probs, k=min(3, NUM_CLASSES), labels=np.arange(NUM_CLASSES))),
        "macro_f1": float(report_dict["macro avg"]["f1-score"]),
        "weighted_f1": float(report_dict["weighted avg"]["f1-score"]),
        "per_class_accuracy": {name: float(per_class_acc[i]) for i, name in enumerate(CLASS_NAMES)},
        "group_accuracy": {str(g): float(sub["correct"].mean()) for g, sub in pred_frame.groupby("source_group")},
        "classification_report": report_dict, "confusion_matrix": cm.tolist(),
        "major_accuracy": float(accuracy_score(major_true, major_pred)),
        "major_prob_accuracy": float(accuracy_score(major_true, major_prob_pred)),
        "major_report": major_report, "major_confusion_matrix": major_cm.tolist(),
        "major_class_names": MAJOR_NAMES, "class_names": CLASS_NAMES,
    }
    with open(RESULT_DIR / f"{model_key}_metrics.json", "w") as f:
        json.dump(jsonable(metrics), f, indent=2)
    pd.DataFrame(report_dict).T.to_csv(RESULT_DIR / f"{model_key}_classification_report.csv")
    pd.DataFrame(major_report).T.to_csv(RESULT_DIR / f"{model_key}_major_report.csv")
    mis = pred_frame[~pred_frame["correct"].astype(bool)].copy()
    mis.to_csv(RESULT_DIR / f"{model_key}_misclassified.csv", index=False)
    print(f"{model_key}: acc {metrics['accuracy']*100:.2f}% | loss {metrics['loss']:.4f} | "
          f"macroF1 {metrics['macro_f1']:.4f} | top3 {metrics['top3_accuracy']*100:.2f}% | "
          f"3-class {metrics['major_accuracy']*100:.2f}%")
    return metrics, pred_frame


def plot_training_curves(history, title: str, output_path: Path):
    frame = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].plot(frame["epoch"], frame["train_loss"], marker="o", ms=3, label="Train")
    axes[0].plot(frame["epoch"], frame["val_loss"], marker="o", ms=3, label="Validation")
    axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("epoch")
    axes[1].plot(frame["epoch"], frame["train_acc"], marker="o", ms=3, label="Train")
    axes[1].plot(frame["epoch"], frame["val_acc"], marker="o", ms=3, label="Validation")
    axes[1].set_title("Accuracy"); axes[1].legend(); axes[1].set_xlabel("epoch")
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(output_path, bbox_inches="tight")
    plt.show()
    plt.close(fig)


## 7. Training K-Fold: 4 model × N_SPLITS fold

Setiap kombinasi (model, fold): train pada ±72% data, validasi ±8% untuk early stopping + pemilihan checkpoint, evaluasi pada test fold ±20% yang tidak pernah dilihat. Estimasi biaya: 4 × 3 = 12 training @ ≤50 epoch. Gunakan GPU. Uji cepat: `TRASH_REPRO_QUICK=1` (2 fold, 2 epoch, tanpa pretrain); lebih teliti: `KFOLD_SPLITS=5` (20 training).

In [ ]:
fold_rows = []
fold_histories = {}  # (model_key, fold) -> history list
runs = {}  # model_key_base -> {label, arch, config, folds: [...]}

for model_idx, (name, spec) in enumerate(MODEL_SPECS.items()):
    print("=" * 80)
    print(f"Model {model_idx+1}/{len(MODEL_SPECS)}: {spec['label']} ({name})")
    print("=" * 80)
    per_fold = []
    for fold, (trainval_idx, test_idx) in enumerate(folds):
        tag = f"{name}_f{fold}"
        print(f"--- fold {fold+1}/{N_SPLITS} [{tag}] ---")
        seed_everything(SEED + fold)
        test_frame = df.iloc[test_idx].reset_index(drop=True)
        trainval_frame = df.iloc[trainval_idx].reset_index(drop=True)
        tr_idx, va_idx = train_test_split(
            np.arange(len(trainval_frame)), test_size=0.10,
            random_state=SEED + fold, stratify=trainval_frame["label"].to_numpy(),
        )
        train_frame = trainval_frame.iloc[tr_idx].reset_index(drop=True)
        val_frame = trainval_frame.iloc[va_idx].reset_index(drop=True)
        print(f"  train {len(train_frame):,} | val {len(val_frame):,} | test {len(test_frame):,}")
        train_loader = make_loader(train_frame, training=True, batch_size=BATCH_SIZE)
        val_loader = make_loader(val_frame, training=False, batch_size=BATCH_SIZE)
        test_loader = make_loader(test_frame, training=False, batch_size=TEST_BATCH_SIZE)
        model = spec["builder"](NUM_CLASSES, pretrained=PRETRAINED).to(DEVICE)
        lr = spec.get("learning_rate", LEARNING_RATE)
        momentum = spec.get("momentum", MOMENTUM)
        weight_decay = spec.get("weight_decay", 1e-4)
        label_smoothing = spec.get("label_smoothing", 0.1)
        criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)
        scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
        checkpoint_path = MODEL_DIR / f"{tag}_trash30_best.pt"
        full_model_path = MODEL_DIR / f"{tag}_trash30_full.pt"
        history_path = RESULT_DIR / f"{tag}_history.csv"
        history, best_val_acc, best_epoch = train_model(
            model=model, train_loader=train_loader, val_loader=val_loader,
            criterion=criterion, optimizer=optimizer, scheduler=scheduler,
            epochs=NUM_EPOCHS, device=DEVICE,
            checkpoint_path=checkpoint_path, full_model_path=full_model_path, history_path=history_path,
            model_name=tag, arch=spec["arch"],
            extra_config={"dataset": "dedupe_gui_curated_kfold", "fold": fold, "n_splits": N_SPLITS,
                            "learning_rate": lr, "momentum": momentum, "weight_decay": weight_decay,
                            "label_smoothing": label_smoothing, "scheduler": "cosine",
                            "augmentation": spec.get("augmentation", "randomresizedcrop+flip")},
            early_stopping_patience=EARLY_STOP_PATIENCE,
            early_stop_min_epochs=EARLY_STOP_MIN_EPOCHS,
        )
        fold_histories[(name, fold)] = history
        plot_training_curves(history, f"{spec['label']} — fold {fold+1}/{N_SPLITS}", FIGURE_DIR / f"{tag}_curves.png")
        eval_model = spec["builder"](NUM_CLASSES, pretrained=False).to(DEVICE)
        ckpt = load_checkpoint(checkpoint_path, DEVICE)
        eval_model.load_state_dict(ckpt["state_dict"])
        eval_model.eval()
        eval_criterion = nn.CrossEntropyLoss()
        metrics, _ = evaluate_and_save(eval_model, test_loader, eval_criterion, DEVICE, test_frame, tag)
        major = metrics.get("major_report", {}) or {}
        row = {"Model": spec["label"], "model_key": name, "fold": fold,
               "Test acc (%)": metrics["accuracy"] * 100, "Loss": metrics["loss"],
               "Top-3 (%)": metrics["top3_accuracy"] * 100, "Macro F1 (30)": metrics["macro_f1"],
               "Macro F1 (3 major)": (major.get("macro avg") or {}).get("f1-score"),
               "3-class acc (%)": metrics["major_accuracy"] * 100,
               "3-class prob acc (%)": metrics["major_prob_accuracy"] * 100,
               "Best epoch": best_epoch, "Best val acc": best_val_acc,
               "Params (M)": spec.get("params_m"), "GMACs": spec.get("gmacs"),
               "LR": lr, "n_test": metrics["num_test"]}
        fold_rows.append(row)
        per_fold.append({"fold": fold, "tag": tag, "metrics": metrics,
                         "best_epoch": best_epoch, "best_val_acc": best_val_acc,
                         "checkpoint": str(checkpoint_path)})
        print()
    runs[name] = {"label": spec["label"], "arch": spec["arch"],
                  "config": {"learning_rate": spec["learning_rate"], "momentum": spec["momentum"],
                             "weight_decay": spec["weight_decay"], "label_smoothing": spec["label_smoothing"],
                             "scheduler": "cosine", "augmentation": spec["augmentation"]},
                  "params_m": spec.get("params_m"), "gmacs": spec.get("gmacs"),
                  "folds": per_fold}

print("Selesai K-Fold training:", list(runs.keys()))


## 8. Agregasi K-Fold: mean ± std, peringkat, dan plot error-bar

Pemenang ditentukan dari **mean test accuracy lintas fold** (std sebagai tie-breaker: std lebih kecil = lebih stabil). Semua artefak memakai prefix `kfold_dedupe_gui_4models` agar tidak menimpa run sebelumnya.

In [ ]:
fold_df = pd.DataFrame(fold_rows)
fold_csv = RESULT_DIR / "kfold_dedupe_gui_4models_fold_metrics.csv"
fold_df.to_csv(fold_csv, index=False)
print("Saved:", fold_csv)
display(fold_df.round(4))

agg = (fold_df.groupby(["Model", "model_key"], as_index=False)
       .agg(mean_acc=("Test acc (%)", "mean"), std_acc=("Test acc (%)", "std"),
            mean_macroF1=("Macro F1 (30)", "mean"), std_macroF1=("Macro F1 (30)", "std"),
            mean_major=("3-class acc (%)", "mean"), std_major=("3-class acc (%)", "std"),
            mean_top3=("Top-3 (%)", "mean"), mean_loss=("Loss", "mean"),
            mean_epoch=("Best epoch", "mean"), n_folds=("fold", "count"),
            params_m=("Params (M)", "first"), gmacs=("GMACs", "first"), lr=("LR", "first")))
agg = agg.sort_values(["mean_acc", "std_acc"], ascending=[False, True]).reset_index(drop=True)
agg["rank"] = np.arange(1, len(agg) + 1)
summary_csv = RESULT_DIR / "kfold_dedupe_gui_4models_summary.csv"
agg.to_csv(summary_csv, index=False)
print("Saved:", summary_csv)
display(agg.round(4))

winner = agg.iloc[0]
print(f"Pemenang sementara (mean acc tertinggi): {winner['Model']} — "
      f"{winner['mean_acc']:.2f}% ± {winner['std_acc']:.2f}% (n={int(winner['n_folds'])})")

# Plot 1: mean test accuracy ± std (error bar), diurutkan peringkat.
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
order = agg.sort_values("mean_acc", ascending=False).reset_index(drop=True)
x = np.arange(len(order))
short = [m.split(" (")[0] for m in order["Model"]]
axes[0].bar(x, order["mean_acc"].astype(float).values, yerr=order["std_acc"].astype(float).values,
             capsize=4, color="#4C78A8")
axes[0].set_xticks(x); axes[0].set_xticklabels(short, rotation=20, ha="right", fontsize=9)
axes[0].set_title(f"Mean test accuracy ± std ({N_SPLITS}-fold, dedupe_gui)")
axes[0].set_ylabel("accuracy (%)")
for i, r in order.iterrows():
    axes[0].text(i, float(r['mean_acc']) + float(r['std_acc'] or 0) + 0.15,
                   f"{float(r['mean_acc']):.2f}±{float(r['std_acc'] or 0):.2f}", ha="center", fontsize=8)
axes[1].bar(x, order["mean_macroF1"].astype(float).values, yerr=order["std_macroF1"].astype(float).values,
             capsize=4, color="#54A24B")
axes[1].set_xticks(x); axes[1].set_xticklabels(short, rotation=20, ha="right", fontsize=9)
axes[1].set_title(f"Mean macro-F1 (30 kelas) ± std ({N_SPLITS}-fold)")
for i, r in order.iterrows():
    axes[1].text(i, float(r['mean_macroF1']) + float(r['std_macroF1'] or 0) + 0.005,
                   f"{float(r['mean_macroF1']):.4f}", ha="center", fontsize=8)
fig.suptitle(f"K-Fold ({N_SPLITS}-fold, stratified, seed {SEED}) — 4 model cosine, dataset dedupe_gui")
fig.tight_layout()
fig_path = FIGURE_DIR / "kfold_dedupe_gui_4models_mean_comparison.png"
fig.savefig(fig_path, bbox_inches="tight")
plt.show()
plt.close(fig)
print("Saved:", fig_path)

# Plot 2: kurva val-acc per model lintas fold (satu gambar per model).
for name, spec in MODEL_SPECS.items():
    fig, ax = plt.subplots(figsize=(10, 4.5))
    for fold in range(N_SPLITS):
        hist = pd.DataFrame(fold_histories.get((name, fold), []))
        if hist.empty:
            continue
        ax.plot(hist["epoch"], hist["val_acc"], marker="o", ms=3, alpha=0.7, label=f"fold {fold+1}")
    ax.set_title(f"{spec['label']} — val accuracy per fold")
    ax.set_xlabel("epoch"); ax.set_ylabel("val acc"); ax.legend(ncols=N_SPLITS)
    fig.tight_layout()
    p = FIGURE_DIR / f"{name}_valcurves_allfolds.png"
    fig.savefig(p, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Saved:", p)


## 9. Simpan ringkasan K-Fold

In [ ]:
summary = {
    "run": "kfold_4models_dedupe_gui",
    "dataset": {
        "curation": "dedupe_gui (duplicate_groups_with_decision.csv / keep_decisions.json)",
        "split_manifest": str(SPLIT_ALL),
        "num_images": int(len(df)),
        "n_splits": int(N_SPLITS),
        "protocol": "StratifiedKFold(shuffle, seed 42) on 30-class label; per outer fold test=1/K, trainval=(K-1)/K split stratified 90/10 into train/val for early stopping",
    },
    "recipe_note": "Cosine recipe per model (LR: RepViT/StarNet-S2 2e-3, MNV2/MNV3-Small 1e-3; SGD momentum 0.9, CosineAnnealingLR T_max=NUM_EPOCHS, ls 0.1, wd 1e-4). Tanpa MobileNetV3-Large TV.",
    "training_defaults": {
        "optimizer": "SGD", "momentum": MOMENTUM,
        "scheduler": "cosine (CosineAnnealingLR, T_max=NUM_EPOCHS)",
        "label_smoothing": 0.1, "weight_decay": 1e-4,
        "augmentation": "randomresizedcrop+flip",
        "epochs": NUM_EPOCHS,
        "early_stopping_patience": EARLY_STOP_PATIENCE,
        "early_stop_min_epochs": EARLY_STOP_MIN_EPOCHS,
        "batch_size": BATCH_SIZE,
        "test_batch_size": TEST_BATCH_SIZE,
        "img_size": IMG_SIZE,
        "seed": SEED,
        "n_splits": N_SPLITS,
    },
    "models": {name: {
        "label": runs[name]["label"], "arch": runs[name]["arch"],
        "config": runs[name].get("config"),
        "params_m": runs[name].get("params_m"), "gmacs": runs[name].get("gmacs"),
        "folds": [{"fold": f["fold"], "tag": f["tag"],
                   "best_epoch": f["best_epoch"], "best_val_acc": f["best_val_acc"],
                   "metrics": {"accuracy": f["metrics"]["accuracy"], "loss": f["metrics"]["loss"],
                               "top3_accuracy": f["metrics"]["top3_accuracy"],
                               "macro_f1": f["metrics"]["macro_f1"],
                               "major_accuracy": f["metrics"]["major_accuracy"]},
                   "checkpoint": f["checkpoint"]} for f in runs[name]["folds"]],
    } for name in runs},
    "aggregate_csv": str(RESULT_DIR / "kfold_dedupe_gui_4models_summary.csv"),
    "fold_csv": str(RESULT_DIR / "kfold_dedupe_gui_4models_fold_metrics.csv"),
    "note": "Tanpa MobileNetV3-Large TV. Mean±std lintas fold = dasar pemilihan model final (mean tertinggi, std terkecil sebagai tie-breaker).",
    "versions": {
        "python": sys.version.split()[0],
        "torch": torch.__version__,
        "torchvision": __import__('torchvision').__version__,
        "timm": __import__('timm').__version__,
    },
}
summary_path = RESULT_DIR / "kfold_dedupe_gui_4models_summary.json"
with open(summary_path, "w") as f:
    json.dump(jsonable(summary), f, indent=2)
print("Saved:", summary_path)
print("Done.")


## 10. Cara menjalankan & keluaran

- **Full run (GPU, 4 model × 3 fold = 12 training):** jalankan `Run All` di GPU (Kaggle/Colab/server). Estimasi lebih berat dari run single-split — pastikan sesi GPU stabil dan checkpoint tersimpan berkala (sudah otomatis per fold).
- **Run lebih teliti (opsional):** set `KFOLD_SPLITS=5` untuk 5-fold (20 training). Smoke test `TRASH_REPRO_QUICK=1` (otomatis 2 fold, 2 epoch, tanpa pretrain — hanya untuk memastikan notebook jalan, **bukan hasil yang dilaporkan**).
- **Keluaran baru (prefix `kfold_dedupe_gui_`, tidak menimpa run lama):** per-fold `*_history.csv`, `*_metrics.json`, `*_predictions.csv`, `*_classification_report.csv`, `*_major_report.csv`, `*_misclassified.csv`, checkpoint `*_trash30_best.pt`/`*_full.pt`, kurva `*_curves.png`, `kfold_dedupe_gui_fold_assignment.csv`, `kfold_dedupe_gui_4models_fold_metrics.csv`, `kfold_dedupe_gui_4models_summary.csv`, `kfold_dedupe_gui_4models_mean_comparison.png`, `*_valcurves_allfolds.png`, `kfold_dedupe_gui_4models_summary.json`.
- **Yang dilaporkan:** tabel mean ± std (agregasi) + ranking + pemenang sementara (mean tertinggi, std terkecil tie-breaker). Per-fold detail ada di CSV/JSON bila reviewer meminta.
